# Oracle routing analysis**Can per-sample modality routing beat simply always using the best fixedconfiguration?** If it cannot, the adaptive-routing contribution does not exist,and this notebook says so cheaply — before any gate is built.The modality ablation established the two tiers:| tier | configuration | GFLOPs | DBA ||---|---|---|---|| cheap | GPS | 0.39 | 0.7298 || escalate | GPS + Camera | 48.2 | 0.8835 |A 124× cost ratio with +0.154 DBA of headroom. The question is how much of thatheadroom a router can actually capture.## What this computes1. **Oracle ceiling.** Escalate only where the cheap tier is wrong *and* the   expensive tier is right. Uses ground truth, so it is unachievable — it is the   upper bound on any router.2. **Realisable gate.** Escalate the least-confident fraction, using only the   cheap model's own output. Sweeping the threshold traces an   accuracy-versus-compute curve that a real system could follow.3. **The decisive comparison.** Does that curve pass above the *always   GPS + Camera* point? Reaching the same DBA at materially less average compute   is the contribution; needing ~all samples escalated means there is none.4. **Whether difficulty is the right gate signal.** AUC of the target-derived   ambiguity measures for predicting "cheap suffices", against the AUC of the   cheap model's own confidence. Difficulty is computed from the power vector and   is **offline only** — a deployed gate cannot see it.No training. A GPU is used only for step 0, turning checkpoints into per-samplepredictions.## What to attachThe ablation checkpoints, as a Kaggle dataset, plus the preprocessed dataset.Checkpoints are found by walking `/kaggle/input`, in either layout:- `.../<slug>/best.pt` — a directory named for the configuration, e.g.  `runs/gps_image/best.pt`- `.../<slug>.pt` — the file itself renamed, e.g. `gps_image.pt`**Do not flatten eight files all called `best.pt` into one folder** — they wouldcollide and the configuration would be unrecoverable. Either keep the`runs/<slug>/` directories or rename to `<slug>.pt`.At minimum the cheap and one expensive configuration must be present. Anythingmissing is reported and skipped.

## 1. Configuration

In [ ]:
from pathlib import Path# The two tiers, as the ablation identified them. Slugs match the run directories.CHEAP = "gps"                  # GPS only          0.39 GFLOPsEXPENSIVE = "gps_image"        # GPS + Camera     48.24 GFLOPsSPLIT = "val"                  # the labelled split with real n (2,198)SECONDARY_SPLIT = "adaptation" # 100 samples, includes the unseen scenario 31BATCH_SIZE = 32WORKERS = 2TOPK = 5OUT = Path("/kaggle/working/outputs/oracle_routing")(OUT / "plots").mkdir(parents=True, exist_ok=True)(OUT / "predictions").mkdir(parents=True, exist_ok=True)# GFLOPs per configuration, measured during the ablation. Used for the cost model.GFLOPS = {    "gps": 0.3869, "gps_radar": 23.5668, "gps_lidar": 23.0530,    "gps_radar_lidar": 46.2328, "gps_image": 48.2398,    "gps_image_lidar": 70.9058, "gps_radar_image": 71.4196,    "gps_radar_image_lidar": 94.0856,}DBA_DELTA = 5.0                # the Delta in eqs. (38)-(39)print(f"cheap tier     {CHEAP:<24} {GFLOPS[CHEAP]:>7.2f} GFLOPs")print(f"escalate to    {EXPENSIVE:<24} {GFLOPS[EXPENSIVE]:>7.2f} GFLOPs")print(f"cost ratio     {GFLOPS[EXPENSIVE] / GFLOPS[CHEAP]:.0f}x")print(f"outputs ->     {OUT}")

## 2. Environment

In [ ]:
import osimport socketimport subprocessimport sysimport torchprint(f"torch {torch.__version__}   cuda: {torch.cuda.is_available()}")if torch.cuda.is_available():    print(f"GPU: {torch.cuda.get_device_properties(0).name}")else:    print("No GPU. Prediction will be slow; the analysis itself needs none.")def has_internet(host="github.com", port=443, timeout=6):    try:        socket.create_connection((host, port), timeout=timeout).close()        return True    except OSError:        return Falseassert has_internet(), "Internet is off; the code cannot be cloned."print("internet: on")

## 3. Dataset and code

In [ ]:
import shutilimport pandas as pdINPUT_ROOT = Path("/kaggle/input")SOURCES = ("development", "adaptation", "test")def walk_dirs(base, follow=True):    for dirpath, dirnames, filenames in os.walk(base, followlinks=follow):        yield Path(dirpath), dirnames, filenamesdef find_index(base):    for cand in (base / "index" / "index", base / "index", base):        if (cand / "samples.csv").exists():            return cand    for d, _dirs, files in walk_dirs(base):        if "samples.csv" in files:            return d    return Nonedef find_source(base, name):    for cand in (base / name / name, base / name):        if cand.is_dir() and any(cand.glob("scenario*")):            return cand    return NoneINDEX_SRC = find_index(INPUT_ROOT)assert INDEX_SRC is not None, "samples.csv not found under /kaggle/input"ROOTS = [INDEX_SRC.parent, INDEX_SRC.parent.parent, INDEX_SRC]SOURCE_DIRS = {s: next((d for d in (find_source(r, s) for r in ROOTS) if d), None)               for s in SOURCES}for s in [k for k, v in SOURCE_DIRS.items() if v is None]:    for d, dn, _f in walk_dirs(INPUT_ROOT):        if d.name == s and any(n.startswith("scenario") for n in dn):            SOURCE_DIRS[s] = d            breakSTAGE = Path("/kaggle/working/data/processed_amber")STAGE.mkdir(parents=True, exist_ok=True)INDEX_DIR = STAGE / "index"if not (INDEX_DIR / "samples.csv").exists():    shutil.copytree(INDEX_SRC, INDEX_DIR, dirs_exist_ok=True)for name, src in SOURCE_DIRS.items():    if src is not None and not (STAGE / name).exists():        (STAGE / name).symlink_to(src)REPO = "https://github.com/SathishAdithiyaaSV/MultiModal-Beam-Prediction.git"PROJECT = Path("/kaggle/working/project")if not (PROJECT / "amber" / "model.py").exists():    r = subprocess.run(["git", "clone", "--depth", "1", REPO, str(PROJECT)],                       capture_output=True, text=True,                       env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})    assert r.returncode == 0, (r.stderr or "")[:400]os.chdir(PROJECT)if str(PROJECT) not in sys.path:    sys.path.insert(0, str(PROJECT))from amber.ablation import config_label, parse_modalitiesfrom amber.config import TrainConfigfrom amber.difficulty import load_with_difficultyfrom amber.predict import load_checkpoint, predict_splitfrom amber.train import pick_deviceprint(f"index {INDEX_DIR}")print(f"code  {PROJECT}")

## 4. Find the ablation checkpointsConfiguration is taken from the run directory name, since checkpoints writtenbefore this notebook existed record only the global modality list. Newer onescarry `enabled_modalities` and that is preferred when present.

In [ ]:
KNOWN = set(GFLOPS)def discover_checkpoints():    '''Locate one checkpoint per configuration, tolerating either upload layout.    Accepts both shapes, since flattening eight files all named best.pt into one    folder would collide:        .../<slug>/best.pt      directory named for the configuration        .../<slug>.pt           file renamed to the configuration    '''    found = {}    for d, _dirs, files in walk_dirs(INPUT_ROOT):        # layout A: a directory named for the configuration        if "best.pt" in files and d.name in KNOWN:            found.setdefault(d.name, d / "best.pt")        # layout B: files renamed to <slug>.pt        for name in files:            if name.endswith(".pt") and name[:-3] in KNOWN:                found.setdefault(name[:-3], d / name)    return foundCKPTS = discover_checkpoints()print(f"{len(CKPTS)}/8 checkpoints found\n")for slug in GFLOPS:    if slug in CKPTS:        print(f"  {slug:<24} {CKPTS[slug]}")    else:        print(f"  {slug:<24} -- missing")missing_core = [s for s in (CHEAP, EXPENSIVE) if s not in CKPTS]if missing_core:    raise SystemExit(        f"The core tiers {missing_core} are missing, so the central question cannot "        f"be answered. Attach the ablation checkpoints -- part 1 supplies "        f"'{CHEAP}', part 2 supplies '{EXPENSIVE}'.")

## 5. Per-sample predictionsThe only GPU step. `--modalities` is essential: without it a configuration'scheckpoint would be scored against whatever modalities each sample happens tohave, silently evaluating a different model. Results are cached, so re-runningis free.

In [ ]:
import gcdevice = pick_device("auto")train_cfg = TrainConfig()def predictions_for(slug, split):    '''Per-sample predictions for one configuration, cached to disk.'''    cache = OUT / "predictions" / f"{split}_{slug}.csv"    if cache.exists():        return pd.read_csv(cache)    mods = parse_modalities(slug.split("_"))    model, cfg = load_checkpoint(CKPTS[slug], device)    frame, _metrics = predict_split(model, INDEX_DIR, split, cfg, train_cfg, device,                                    BATCH_SIZE, WORKERS, TOPK, modalities=mods)    frame["slug"] = slug    frame.to_csv(cache, index=False)    del model    gc.collect()    if device.type == "cuda":        torch.cuda.empty_cache()    print(f"  {config_label(mods):<32} {len(frame)} samples -> {cache.name}")    return framepreds = {}for split in (SPLIT, SECONDARY_SPLIT):    print(f"\n{split}:")    preds[split] = {s: predictions_for(s, split) for s in CKPTS}print("\ndone")

## 6. Per-sample correctness and DBADBA is an average of per-sample terms, so it decomposes exactly:    per-sample DBA = mean over k=1..3 of ( 1 - min( min_{k'<=k} |y - yhat_k'| / Delta , 1 ) )which lets a routed mixture be scored without re-running any model.

In [ ]:
import numpy as npdef per_sample_scores(frame, topk=3, delta=DBA_DELTA):    '''Adds correct@1, correct@3 and a per-sample DBA contribution.'''    f = frame.copy()    beams = np.stack([f[f"top{k}_beam"].to_numpy() for k in range(1, topk + 1)], axis=1)    true = f["true_beam"].to_numpy()[:, None]    f["correct1"] = (beams[:, 0] == true[:, 0]).astype(int)    f["correct3"] = (beams[:, :3] == true).any(axis=1).astype(int)    dist = np.abs(beams - true) / delta    best = np.minimum.accumulate(dist, axis=1).clip(max=1.0)    f["dba"] = (1.0 - best).mean(axis=1)    return fdef build_matrix(split):    '''sample_id x configuration, for correctness, DBA and cheap-tier confidence.'''    parts = []    for slug, frame in preds[split].items():        f = per_sample_scores(frame)        parts.append(f[["sample_id", "scenario", "true_beam", "top1_beam", "top1_conf",                        "correct1", "correct3", "dba"]].assign(slug=slug))    long = pd.concat(parts, ignore_index=True)    wide = long.pivot_table(index=["sample_id", "scenario"], columns="slug",                            values=["correct1", "correct3", "dba", "top1_conf"])    wide.columns = [f"{a}__{b}" for a, b in wide.columns]    return long, wide.reset_index()long_val, M = build_matrix(SPLIT)diff = load_with_difficulty(INDEX_DIR)M = M.merge(diff[["sample_id", "margin_db", "entropy_bits", "n_within_10pct",                  "spread_db"]], on="sample_id", how="left")M.to_csv(OUT / f"per_sample_{SPLIT}.csv", index=False)print(f"{len(M)} samples x {len(CKPTS)} configurations\n")print("sanity check -- per-sample means must reproduce the ablation's aggregates:")for slug in CKPTS:    print(f"  {slug:<24} Top-1 {M[f'correct1__{slug}'].mean():.4f}   "          f"Top-3 {M[f'correct3__{slug}'].mean():.4f}   "          f"DBA {M[f'dba__{slug}'].mean():.4f}")

## 7. Oracle ceilingHow much headroom exists at all? Partition the samples by what the two tiers getright.

In [ ]:
c_ok = M[f"correct1__{CHEAP}"].astype(bool)e_ok = M[f"correct1__{EXPENSIVE}"].astype(bool)quad = pd.Series({    "both correct": int((c_ok & e_ok).sum()),    "cheap only": int((c_ok & ~e_ok).sum()),    "expensive only (escalation pays)": int((~c_ok & e_ok).sum()),    "neither": int((~c_ok & ~e_ok).sum()),})print("Top-1 agreement between the tiers:\n")print((quad / len(M) * 100).round(1).astype(str).add(" %").to_string())print(f"\nn = {len(M)}")cheap_acc, exp_acc = c_ok.mean(), e_ok.mean()oracle_acc = (c_ok | e_ok).mean()# escalate only where it is both necessary and sufficientescalate_need = (~c_ok & e_ok)f_oracle = escalate_need.mean()cost_oracle = GFLOPS[CHEAP] + f_oracle * (GFLOPS[EXPENSIVE] - GFLOPS[CHEAP])print(f"\nalways cheap        Top-1 {cheap_acc:.4f}   {GFLOPS[CHEAP]:>6.2f} GFLOPs")print(f"always expensive    Top-1 {exp_acc:.4f}   {GFLOPS[EXPENSIVE]:>6.2f} GFLOPs")print(f"ORACLE router       Top-1 {oracle_acc:.4f}   {cost_oracle:>6.2f} GFLOPs "      f"(escalates {f_oracle*100:.1f}% of samples)")print(f"\nThe oracle beats always-expensive by {oracle_acc - exp_acc:+.4f} Top-1 at "      f"{cost_oracle / GFLOPS[EXPENSIVE] * 100:.0f}% of its compute.")print("It is an unachievable upper bound -- it consults the ground truth to decide.")

## 8. Realisable gateEscalate the least-confident samples, judged only by the cheap model's ownoutput — information a deployed system actually has. Sweeping the thresholdtraces the accuracy-versus-compute curve.

In [ ]:
conf = M[f"top1_conf__{CHEAP}"].to_numpy()order = np.argsort(conf)            # least confident firstn = len(M)c1 = M[f"correct1__{CHEAP}"].to_numpy()e1 = M[f"correct1__{EXPENSIVE}"].to_numpy()cd = M[f"dba__{CHEAP}"].to_numpy()ed = M[f"dba__{EXPENSIVE}"].to_numpy()rows = []for f in np.linspace(0, 1, 101):    k = int(round(f * n))    esc = np.zeros(n, bool)    esc[order[:k]] = True    rows.append({        "escalated_frac": f,        "Top-1": np.where(esc, e1, c1).mean(),        "DBA": np.where(esc, ed, cd).mean(),        "GFLOPs": GFLOPS[CHEAP] + f * (GFLOPS[EXPENSIVE] - GFLOPS[CHEAP]),    })sweep = pd.DataFrame(rows)sweep.to_csv(OUT / "confidence_gate_sweep.csv", index=False)# the decisive numbers: where does the gate match always-expensive?def first_reaching(col, target):    hit = sweep[sweep[col] >= target]    return hit.iloc[0] if len(hit) else Noneprint(f"always expensive: Top-1 {exp_acc:.4f}  DBA {ed.mean():.4f}  "      f"{GFLOPS[EXPENSIVE]:.2f} GFLOPs\n")for col, target in (("Top-1", exp_acc), ("DBA", ed.mean())):    r = first_reaching(col, target)    if r is None:        print(f"  the gate never reaches always-expensive {col} "              f"(best {sweep[col].max():.4f}) -- routing cannot match it")    else:        save = 1 - r.GFLOPs / GFLOPS[EXPENSIVE]        print(f"  matches its {col:<6} at {r.escalated_frac*100:5.1f}% escalated, "              f"{r.GFLOPs:6.2f} GFLOPs -- {save*100:.0f}% compute saved")for frac in (0.25, 0.5):    r = sweep.iloc[(sweep.escalated_frac - frac).abs().idxmin()]    print(f"\n  at {frac*100:.0f}% escalated: Top-1 {r['Top-1']:.4f} "          f"({r['Top-1']/exp_acc*100:.1f}% of always-expensive), DBA {r.DBA:.4f} "          f"({r.DBA/ed.mean()*100:.1f}%), {r.GFLOPs:.2f} GFLOPs")

## 9. Is confidence the right gate signal?AUC for predicting "the cheap tier is already correct". The target-deriveddifficulty measures are included as a reference ceiling — a deployed gate cannotread them, since they are computed from the answer.

In [ ]:
def auc(score, label):    '''Rank-based AUC, no sklearn dependency.'''    label = np.asarray(label).astype(bool)    if label.all() or not label.any():        return float("nan")    s = np.asarray(score, dtype=float)    ok = np.isfinite(s)    s, label = s[ok], label[ok]    r = pd.Series(s).rank().to_numpy()    n1, n0 = label.sum(), (~label).sum()    return (r[label].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)target = M[f"correct1__{CHEAP}"].astype(bool)sigs = {    f"cheap model confidence ({CHEAP})": M[f"top1_conf__{CHEAP}"],    "margin_db (offline only)": M["margin_db"],    "entropy_bits (offline only, negated)": -M["entropy_bits"],    "n_within_10pct (offline only, negated)": -M["n_within_10pct"],    "spread_db (offline only)": M["spread_db"],}res = pd.DataFrame([{"signal": k, "AUC": auc(v, target)} for k, v in sigs.items()]                   ).sort_values("AUC", ascending=False)res.to_csv(OUT / "gate_signal_auc.csv", index=False)print("predicting 'cheap tier is already correct':\n")print(res.round(4).to_string(index=False))print("\n0.5 = no signal. The offline rows bound what a perfect difficulty-aware")print("gate could do; only the confidence row is realisable at inference.")

## 10. Plots

In [ ]:
import matplotlib.pyplot as pltP = OUT / "plots"fig, ax = plt.subplots(1, 2, figsize=(15, 4.8))for a, col, base in ((ax[0], "Top-1", exp_acc), (ax[1], "DBA", ed.mean())):    a.plot(sweep.GFLOPs, sweep[col], lw=2, label="confidence gate")    a.scatter([GFLOPS[CHEAP]], [sweep[col].iloc[0]], s=80, c="tab:green",              zorder=5, label=f"always cheap ({CHEAP})")    a.scatter([GFLOPS[EXPENSIVE]], [base], s=80, c="tab:red", zorder=5,              label=f"always expensive ({EXPENSIVE})")    a.axhline(base, color="tab:red", ls=":", lw=1)    if col == "Top-1":        a.scatter([cost_oracle], [oracle_acc], s=110, marker="*", c="k",                  zorder=6, label="oracle ceiling")    a.set(xlabel="average forward GFLOPs per sample", ylabel=col,          title=f"{col} vs inference cost")    a.grid(alpha=.3)    a.legend(frameon=False, fontsize=8)plt.tight_layout()plt.savefig(P / "accuracy_vs_cost.png", dpi=150)plt.show()fig, ax = plt.subplots(1, 2, figsize=(15, 4.2))(quad / len(M) * 100).plot.barh(ax=ax[0], color=["tab:green", "tab:orange",                                                 "tab:blue", "tab:grey"])ax[0].set(xlabel="% of samples", title="Where the two tiers agree (Top-1)")ax[0].grid(alpha=.3, axis="x")ax[0].invert_yaxis()r = res.dropna()ax[1].barh(r.signal, r.AUC, color=["tab:purple" if "confidence" in s else "tab:grey"                                   for s in r.signal])ax[1].axvline(0.5, color="k", lw=.8, ls="--")ax[1].set(xlabel="AUC for 'cheap tier already correct'", xlim=(0.4, 1.0),          title="Gate signal quality")ax[1].grid(alpha=.3, axis="x")ax[1].invert_yaxis()plt.tight_layout()plt.savefig(P / "agreement_and_signals.png", dpi=150)plt.show()print(f"plots -> {P}")

## 11. VerdictGenerated from the measured numbers, including when they say the contribution isnot there.

In [ ]:
L = []gain_over_fixed = oracle_acc - exp_accr_t1 = first_reaching("Top-1", exp_acc)r_dba = first_reaching("DBA", ed.mean())conf_auc = float(res[res.signal.str.contains("confidence")].AUC.iloc[0])best_off = res[res.signal.str.contains("offline")].AUC.max()L.append(f"**Headroom.** {quad['expensive only (escalation pays)'] / len(M) * 100:.1f}% of "         f"samples are wrong under {CHEAP} but right under {EXPENSIVE}, so escalation can "         f"help there. {quad['cheap only'] / len(M) * 100:.1f}% are the reverse -- escalation "         f"actively *hurts* them -- and {quad['neither'] / len(M) * 100:.1f}% are beyond both.")L.append(f"**Oracle ceiling.** A router with perfect foresight reaches Top-1 "         f"{oracle_acc:.4f} at {cost_oracle:.2f} GFLOPs, against {exp_acc:.4f} at "         f"{GFLOPS[EXPENSIVE]:.2f} for always-{EXPENSIVE}: {gain_over_fixed:+.4f} Top-1 "         f"for {(1 - cost_oracle / GFLOPS[EXPENSIVE]) * 100:.0f}% less compute. "         + ("That is real headroom." if gain_over_fixed > 0.005 else            "The accuracy gain is negligible, so routing's case rests entirely on "            "the compute saving."))if r_dba is not None:    L.append(f"**Realisable gate.** Escalating by the cheap model's own confidence "             f"matches always-{EXPENSIVE} DBA at {r_dba.escalated_frac * 100:.0f}% of "             f"samples escalated, i.e. {r_dba.GFLOPs:.2f} against "             f"{GFLOPS[EXPENSIVE]:.2f} GFLOPs -- "             f"**{(1 - r_dba.GFLOPs / GFLOPS[EXPENSIVE]) * 100:.0f}% of the compute "             f"removed at equal DBA**. "             + ("That is a contribution worth building." if r_dba.escalated_frac < 0.8 else                "But it escalates almost everything, so the saving is thin."))else:    L.append(f"**Realisable gate.** The confidence gate never reaches "             f"always-{EXPENSIVE} DBA (best {sweep.DBA.max():.4f} against "             f"{ed.mean():.4f}). **Routing on confidence cannot match the best fixed "             f"configuration**, so on this evidence the adaptive contribution as framed "             f"does not hold up.")L.append(f"**Gate signal.** The cheap model's confidence gives AUC {conf_auc:.3f} for "         f"predicting that the cheap tier already suffices"         + (f"; the best offline difficulty measure reaches {best_off:.3f}, so a "            f"difficulty-aware gate could add at most "            f"{best_off - conf_auc:+.3f} AUC -- and cannot be deployed, since those "            f"measures are computed from the target."            if np.isfinite(best_off) else ".")         + (" Confidence alone is a usable signal." if conf_auc > 0.65 else            " That is weak, which caps how well any confidence gate can work and is "            "the first thing to improve."))L.append("**Caveats.** Measured on the validation split, in-domain by construction. "         "The ablation showed camera generalises badly to the unseen scenario 31, "         "where GPS alone is the better configuration, so a gate tuned here may "         "escalate exactly the wrong way on the 48% of the test split that is "         "scenario 31. Difficulty measures are target-derived and offline only.")text = "\n\n".join(L)print(text)(OUT / "verdict.md").write_text(text)

## 12. Secondary split and outputs

In [ ]:
# the same comparison on adaptation, which contains the unseen scenario 31long_a, Ma = build_matrix(SECONDARY_SPLIT)Ma.to_csv(OUT / f"per_sample_{SECONDARY_SPLIT}.csv", index=False)rows = []for slug in (CHEAP, EXPENSIVE):    for scn, g in Ma.groupby("scenario"):        rows.append({"configuration": slug, "scenario": scn, "n": len(g),                     "Top-1": g[f"correct1__{slug}"].mean(),                     "DBA": g[f"dba__{slug}"].mean()})sec = pd.DataFrame(rows)sec.to_csv(OUT / f"tiers_by_scenario_{SECONDARY_SPLIT}.csv", index=False)print(f"{SECONDARY_SPLIT} -- the two tiers per scenario "      f"(scenario 31 is unseen in training):\n")print(sec.round(4).to_string(index=False))print(f"\n\n{OUT}:")for p in sorted(OUT.rglob("*")):    if p.is_file():        print(f"  {p.relative_to(OUT)}  ({p.stat().st_size / 1e3:.1f} KB)")